<a href="https://colab.research.google.com/github/cc-huang-0716/internship_tests/blob/main/yolov8n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import shutil
import random
from google.colab import drive
import glob
from glob import glob

# 掛載 Drive
drive.mount('/content/drive')
anomaly_zip_path = "/content/drive/MyDrive/anomaly100.zip"
stable_zip_path = "/content/drive/MyDrive/stable500.zip"
extract_dir = "/content/dataset"

# 解壓縮
with zipfile.ZipFile(anomaly_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
with zipfile.ZipFile(stable_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# 設定資料夾名稱
#anomaly_dir = os.path.join(extract_dir, "anomaly_label")
stable_dir = os.path.join(extract_dir, "stable_series_images")

# 建立空白標註檔
for f in os.listdir(stable_dir):
    if f.endswith(".png") or f.endswith(".jpg"):
        txt_name = os.path.splitext(f)[0] + ".txt"
        txt_path = os.path.join(stable_dir, txt_name)
        if not os.path.exists(txt_path):
            open(txt_path, "w").close()

# 各自收集圖片路徑
#anomaly_images = [os.path.join(anomaly_dir, f) for f in os.listdir(anomaly_dir) if f.endswith(".jpg") or f.endswith(".png")]
stable_images  = [os.path.join(stable_dir, f) for f in os.listdir(stable_dir) if f.endswith(".jpg") or f.endswith(".png")]

base_dir = "/content/dataset"
stable_dir = os.path.join(base_dir, "stable_series_images")
merged_dir = os.path.join(base_dir, "merged")
os.makedirs(merged_dir, exist_ok=True)

# 取得空白背景圖與標註
stable_imgs = sorted([f for f in glob(f"{stable_dir}/*") if f.endswith(('.png', '.jpg'))])
random.shuffle(stable_imgs)
N = len(stable_imgs)
stable_train = stable_imgs[:int(N*0.8)]
stable_val   = stable_imgs[int(N*0.8):int(N*0.9)]
stable_test  = stable_imgs[int(N*0.9):]

splits = {
    "train": {"img": glob(f"{base_dir}/train/images/*"), "lbl": glob(f"{base_dir}/train/labels/*"), "stable": stable_train},
    "val": {"img": glob(f"{base_dir}/val/images/*"), "lbl": glob(f"{base_dir}/val/labels/*"), "stable": stable_val},
    "test": {"img": glob(f"{base_dir}/test/images/*"), "lbl": glob(f"{base_dir}/test/labels/*"), "stable": stable_test},
}

# 複製資料與標註
for split in splits:
    img_out = os.path.join(merged_dir, f"images/{split}")
    lbl_out = os.path.join(merged_dir, f"labels/{split}")
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    # 異常圖片與標註
    for img_path in splits[split]["img"]:
        fname = os.path.basename(img_path)
        shutil.copy(img_path, os.path.join(img_out, fname))
        label_path = os.path.join(base_dir, split, "labels", os.path.splitext(fname)[0] + ".txt")
        if os.path.exists(label_path):
            shutil.copy(label_path, os.path.join(lbl_out, os.path.basename(label_path)))

    # 正常圖片與空標註
    for img_path in splits[split]["stable"]:
        fname = os.path.basename(img_path)
        shutil.copy(img_path, os.path.join(img_out, fname))
        empty_txt = os.path.join(lbl_out, os.path.splitext(fname)[0] + ".txt")
        open(empty_txt, "w").close()

# 建立 data.yaml
with open(os.path.join(merged_dir, "data.yaml"), "w") as f:
    f.write(f"""
train: {merged_dir}/images/train
val: {merged_dir}/images/val
test: {merged_dir}/images/test

nc: 1
names: ['anomaly']
""")

print("已建立完成")

In [ ]:
import glob

for path in glob.glob(f"{extract_dir}/labels/*/*.txt"):
    with open(path, "r") as f:
        for i, line in enumerate(f):
            parts = line.strip().split()
            if len(parts) != 5:
                print(f"[格式錯誤] {path} line {i+1}: {line.strip()}")

In [ ]:
import sys
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install'] + 'ultralytics'.split(), check=True)
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data="/content/dataset/data.yaml", epochs=50, imgsz=640)
results = model.predict(
    source='/content/dataset/train/images',
    save=True,
    conf=0.5,
    imgsz=640,
    show=True
    )

In [ ]:
model = YOLO("/content/runs/detect/train/weights/best.pt")
results = model.predict("/content/dataset/valid/images", save=True)

model = YOLO("/content/runs/detect/train/weights/best.pt")
model.train(
    data="dataset/data.yaml",
    epochs=50,
    imgsz=640
)

In [ ]:
results = model.predict("/content/dataset/merged/images/test", save=True)

In [ ]:
metrics = model.val(split="test")
print(metrics.box.map50)
print(metrics.box.map)